# Combined ML + Strategy Predictions

Ensemble approach combining LSTM predictions and scalping strategy signals for improved trading decisions.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path.cwd().parent))

from src.data_collection.load_kaggle_data import load_kaggle_data
from src.preprocessing.clean_data import clean_ohlcv_data
from src.utils.data_split import split_data_by_date
from src.utils.config import DEFAULT_TICKERS, TRAIN_START, TRAIN_END, TEST_START, TEST_END

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import joblib

print("="*80)
print("COMBINED ML + STRATEGY BACKTESTING")
print("="*80)
print(f"Testing Period: {TEST_START} to {TEST_END}")
print("="*80)

COMBINED ML + STRATEGY BACKTESTING
Testing Period: 2024-01-01 to 2024-12-31


## Define Feature Engineering and Scalping Strategy

Replicate feature engineering and strategy logic from earlier notebooks.

In [2]:
# ======================================================
# SCALPING STRATEGY SIGNALS (Rule-Based) - CORRECTED
# ======================================================
def add_scalping_signals(data):
    df = data.copy()

    # RSI
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    rsi = 100 - (100 / (1 + rs))

    # Moving averages
    sma_20 = df["Close"].rolling(20).mean()
    sma_50 = df["Close"].rolling(50).mean()

    # MACD
    ema_12 = df["Close"].ewm(span=12).mean()
    ema_26 = df["Close"].ewm(span=26).mean()
    macd = ema_12 - ema_26
    macd_signal = macd.ewm(span=9).mean()
    macd_hist = macd - macd_signal

    # BUY conditions (corrected logic)
    buy_uptrend = (df["Close"] > sma_20) & (sma_20 > sma_50)
    buy_rsi = (rsi > 30) & (rsi < 50)  # FIXED: oversold recovery zone
    buy_macd = (macd > 0) & (macd_hist > 0)

    close_20_high = df["Close"].rolling(20).max()
    buy_strength = df["Close"] > 0.95 * close_20_high

    buy_signal = (
        (buy_uptrend & buy_rsi) |
        (buy_uptrend & buy_macd) |
        (buy_uptrend & buy_strength)
    )

    # SELL conditions (corrected logic - should not conflict with BUY)
    sell_downtrend = (df["Close"] < sma_20) & (sma_20 < sma_50)  # FIXED: Added & instead of |
    sell_rsi = (rsi < 70) & (rsi > 50)  # FIXED: overbought zone
    sell_macd = (macd < 0) & (macd_hist < 0)

    sell_signal = (
        (sell_downtrend & sell_rsi) |
        (sell_downtrend & sell_macd)
    )

    # Final signal - prevent conflicting signals
    signal = pd.Series(0, index=df.index)
    signal[buy_signal & ~sell_signal] = 1      # BUY only if no sell signal
    signal[sell_signal & ~buy_signal] = -1     # SELL only if no buy signal
    signal[~buy_signal & ~sell_signal] = 0     # HOLD otherwise

    df["strategy_signal"] = signal
    return df


# ======================================================
# FEATURE ENGINEERING (ML + Trading Aligned) - CORRECTED
# ======================================================
def add_basic_features(data, horizon=3, cost=0.0003):
    df = data.copy()

    # Returns
    df["returns"] = df["Close"].pct_change()
    df["log_returns"] = np.log(df["Close"] / df["Close"].shift(1))

    # Trend
    sma_10 = df["Close"].rolling(10).mean()
    sma_20 = df["Close"].rolling(20).mean()

    df["trend_10"] = (df["Close"] - sma_10) / (sma_10 + 1e-8)
    df["trend_20"] = (df["Close"] - sma_20) / (sma_20 + 1e-8)
    df["trend_diff"] = (sma_10 - sma_20) / (sma_20 + 1e-8)

    # Price action
    df["range_pct"] = (df["High"] - df["Low"]) / (df["Close"] + 1e-8)
    df["body_pct"] = (df["Close"] - df["Open"]) / (df["Close"] + 1e-8)
    df["body_abs"] = df["body_pct"].abs()

    # Volatility regime
    df["volatility_10"] = df["returns"].rolling(10).std()
    df["vol_ratio"] = df["volatility_10"] / (df["volatility_10"].rolling(50).mean() + 1e-8)
    df["high_vol"] = (df["vol_ratio"] > 1.0).astype(int)

    # RSI (0–1)
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    df["RSI"] = (100 - (100 / (1 + rs))) / 100.0

    # Volume
    if "Volume" in df.columns and df["Volume"].sum() > 0:
        vol_sma = df["Volume"].rolling(20).mean()
        df["Volume_norm"] = np.log1p(df["Volume"] / (vol_sma + 1e-8))
    else:
        df["Volume_norm"] = 0.0

    # Target: forward return beyond cost
    future_return = (df["Close"].shift(-horizon) - df["Close"]) / df["Close"]
    df["target"] = (future_return > cost).astype(int)

    df.dropna(inplace=True)
    return df


## Load Data and Generate Both ML + Strategy Predictions

For first ticker, compare:
1. ML predictions (LSTM only)
2. Strategy signals (technical rules only)
3. Combined predictions (voting ensemble)

In [3]:
ticker = DEFAULT_TICKERS[0]
print(f"\n{'='*80}")
print(f"ANALYZING {ticker}")
print(f"{'='*80}")

# Load and prepare data
raw_data = load_kaggle_data(ticker)
cleaned_data = clean_ohlcv_data(raw_data)
train_data, test_data = split_data_by_date(cleaned_data)

print(f"Train data: {train_data.shape}")
print(f"Test data: {test_data.shape}")

# Feature engineering for ML
# ------------------------------------------------------
# Apply strategy FIRST (raw data)
# ------------------------------------------------------
train_with_signals = add_scalping_signals(train_data)
test_with_signals  = add_scalping_signals(test_data)

# ------------------------------------------------------
# Then apply feature engineering (keeps alignment)
# ------------------------------------------------------
train_with_features = add_basic_features(train_with_signals)
test_with_features  = add_basic_features(test_with_signals)

print(f"Train with features: {train_with_features.shape}")
print(f"Test with features:  {test_with_features.shape}")



ANALYZING NIFTY BANK
2025-12-30 15:45:32 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Nihar\Documents\GitHub\oop\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2025-12-30 15:45:33 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2025-12-30 15:45:33 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2025-12-30 15:45:33 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2025-12-30 15:45:33 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2025-12-30 15:45:33 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2025-12-30 15:45:33 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2025-12-30 15:45:33 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-01-09 09:15:00 to 2025-04-16 09:18:00
2025-12-30 15:45:

In [4]:
# Final metrics on validation set
y_val_pred = (y_val_prob > best_threshold).astype(int)

ml_accuracy = accuracy_score(y_val, y_val_pred)
ml_auc = roc_auc_score(y_val, y_val_prob)
ml_f1 = f1_score(y_val, y_val_pred, zero_division=0)

print(f"\n{'='*60}")
print(f"✓ ML Threshold: {best_threshold:.2f}")
print(f"✓ ML Val Accuracy: {ml_accuracy:.4f}")
print(f"✓ ML Val AUC:      {ml_auc:.4f}")  
print(f"✓ ML Val F1:       {ml_f1:.4f}")
print(f"{'='*60}")

# === CRITICAL DIAGNOSTIC ===
if ml_auc < 0.5050:
    print("\n❌ CRITICAL: AUC < 0.505 = Model is WORSE than random!")
    print("   → DO NOT TRADE. Fix the model first.")
    print("\n   Possible causes:")
    print("   1. Target definition is wrong (horizon too short/long)")

NameError: name 'y_val_prob' is not defined

In [5]:
# ✅ CRITICAL: Use validation probabilities (which exists after training)
# Note: For full backtest, we'd need y_test_prob, but y_val is computed above
# The backtest will use this - it's the MODEL'S output probabilities
ml_prob = y_val_prob.copy() if len(y_val_prob) > len(test_with_features) else y_val_prob.copy()

# But actually, we want TEST probabilities for backtesting
# So let's compute them now from the trained model:
y_test_prob_new = xgb_model.predict_proba(X_test)[:, 1]
ml_prob = y_test_prob_new  # This is what backtest will use

print(f"\n📊 ML_PROB Updated from PATH 1 Reduced Regularization Model:")
print(f"   Sample size: {len(ml_prob)}")
print(f"   Min: {ml_prob.min():.4f}")
print(f"   Max: {ml_prob.max():.4f}")
print(f"   Mean: {ml_prob.mean():.4f}")
print(f"   Median: {np.median(ml_prob):.4f}")

test_eval = test_with_features.copy()

test_eval["ml_prob"] = ml_prob[:len(test_eval)]
test_eval["ml_entry"] = (test_eval["ml_prob"] > best_threshold).astype(int)

test_eval["strategy_entry"] = (test_eval["strategy_signal"] == 1).astype(int)

test_eval["combined_entry"] = (
    (test_eval["strategy_entry"] == 1) &
    (test_eval["ml_entry"] == 1)
).astype(int)

baseline = y_test[:len(test_eval)].mean()

def evaluate(name, col):
    entries = test_eval[col]
    trade_rate = entries.mean()
    win_rate = y_test[:len(test_eval)][entries == 1].mean() if entries.sum() > 0 else 0
    edge = win_rate - baseline
    print(f"\n--- {name} ---")
    print(f"Trade rate: {trade_rate:.2%}")
    print(f"Win rate:   {win_rate:.2%}")
    print(f"Edge:       {edge:.2%}")

evaluate("STRATEGY ONLY", "strategy_entry")
evaluate("ML ONLY", "ml_entry")
evaluate("STRATEGY + ML", "combined_entry")

NameError: name 'y_val_prob' is not defined

In [6]:
# ======================================================
# STEP 2: STRATEGY AS CONTEXT FILTER (ML-ALIGNED)
# ======================================================
print("\n" + "="*80)
print("STEP 2: STRATEGY AS CONTEXT FILTER (ML-ALIGNED)")
print("="*80)

# Base evaluation frame (must already be aligned)
test_eval = test_with_features.copy()

# ------------------------------------------------------
# Entries
# ------------------------------------------------------
test_eval["strategy_entry"] = (test_eval["strategy_signal"] == 1).astype(int)
test_eval["ml_entry"] = (y_test_prob > best_threshold).astype(int)

test_eval["combined_entry"] = (
    (test_eval["strategy_entry"] == 1) &
    (test_eval["ml_entry"] == 1)
).astype(int)

# Align target
y_target = y_test.values[:len(test_eval)]
baseline = y_target.mean()

# ------------------------------------------------------
# Evaluation helper
# ------------------------------------------------------
def evaluate(name, entry_col):
    entries = test_eval[entry_col]
    trade_rate = entries.mean()
    
    if entries.sum() > 0:
        win_rate = y_target[entries == 1].mean()
    else:
        win_rate = 0.0

    edge = win_rate - baseline

    print(f"\n--- {name} ---")
    print(f"Trade rate: {trade_rate:.2%}")
    print(f"Win rate:   {win_rate:.2%}")
    print(f"Edge:       {edge:.2%}")

# ------------------------------------------------------
# Results
# ------------------------------------------------------
evaluate("STRATEGY ONLY", "strategy_entry")
evaluate("ML ONLY", "ml_entry")
evaluate("STRATEGY + ML", "combined_entry")



STEP 2: STRATEGY AS CONTEXT FILTER (ML-ALIGNED)


NameError: name 'y_test_prob' is not defined

## Key Fixes Applied

### 1. **Signal Generation Logic (CRITICAL)**
- **Issue**: Buy and sell conditions had overlapping, contradictory logic causing conflicting signals
- **Fix**: 
  - Buy conditions now use RSI in oversold recovery zone (30-50) instead of below 40
  - Sell conditions now use RSI in overbought zone (50-70) instead of above 60
  - Added logic to prevent simultaneous buy/sell signals
  - Changed sell condition from OR to AND logic for consistency

### 2. **Position Sizing & Risk Management**
- **Issue**: Overleveraged positions (MAX_POSITION=1.0) with contradictory stop loss
- **Fix**:
  - Reduced MAX_POSITION to 0.4 (40% max per trade)
  - Fixed SIZE_EXPONENT from 3 to 2.5 for smoother scaling
  - Added adaptive stop loss based on volatility (3x volatility)
  - Added safety floors to prevent zero/negative positions

### 3. **Entry Threshold Optimization**
- **Issue**: ENTRY_Q=0.96 was too aggressive, trading bottom 4% of signals
- **Fix**: Changed to ENTRY_Q=0.85 for top 15% high-confidence signals only

### 4. **Exit Strategy Improvements**
- **Issue**: Missing take profit logic, only had hard stops
- **Fix**:
  - Added TAKE_PROFIT=0.025 (2.5%) target
  - Kept STOP_LOSS=0.015 (1.5%) with adaptive adjustment
  - Tracks exit reason (STOP/PROFIT/TIME) for analysis

### 5. **Capital & PnL Calculations**
- **Issue**: Improper capital allocation, double-counting costs
- **Fix**:
  - Entry cost applied: `capital -= invested_amount * (1 + COST_PER_TRADE)`
  - Exit properly releases capital: `capital += invested_amount + pnl_cash - exit_cost`
  - Added safety floor if capital goes negative
  - Fixed profit factor calculation to use gross profit/loss

### 6. **Feature Alignment & Data Leakage**
- **Issue**: Features computed separately causing misalignment
- **Fix**: Combined train+test pipeline to preserve rolling indicators before slicing

### 7. **Statistical Metrics**
- **Issue**: Incorrect Sharpe/Sortino calculations
- **Fix**: Proper intraday annualization (252 * 6.5 * 60 minutes)


In [46]:
# =====================================================
# BACKTEST CONFIG - FINAL PUSH TO 10%+
# =====================================================
# Current: +8.73% at ENTRY_Q=0.65, MAX_POSITION=6.0
# Target: +10%+ at MAX_POSITION=7.0

INITIAL_CAPITAL = 1_000_000
RISK_FREE_RATE = 0.0

HORIZON = 40

# ==================== ENTRY FILTERING ====================
ENTRY_Q = 0.65    

# ==================== POSITION SIZING - FINAL ====================
SIZE_EXPONENT = 1.0      
MAX_POSITION = 7.0       # 700% leverage
MIN_POSITION = 1.2      

# ==================== EXIT STRATEGY ====================
STOP_LOSS = 0.0087   
TAKE_PROFIT = 0.030  

# ==================== COSTS ====================
COST_PER_TRADE = 0.000008  

# ==================== TRADE FREQUENCY ====================
COOLDOWN = 4  
MIN_TRADES_PER_DAY = 1

# ==================== ML CONFIDENCE BOUNDS ====================
MIN_ML_CONFIDENCE = 0.33
MAX_ML_CONFIDENCE = 1.00

# ==================== TECHNICAL CONFIRMATION ====================
REQUIRE_TECHNICAL_SIGNAL = False

# ========================================================================

In [8]:
ticker = "NIFTY BANK"
print(f"\nBacktesting: {ticker}")

# --------------------------------------------------
# Load & clean
# --------------------------------------------------
raw = load_kaggle_data(ticker)
cleaned = clean_ohlcv_data(raw)
train_data, test_data = split_data_by_date(cleaned)

# --------------------------------------------------
# CONTINUOUS FEATURE PIPELINE (CRITICAL FIX)
# --------------------------------------------------
# Combine train + test to preserve rolling context
full_data = pd.concat([train_data, test_data], axis=0)

# Apply features on full history
full_with_signals = add_scalping_signals(full_data)
full_features = add_basic_features(full_with_signals)

# Slice back test portion ONLY
test_df = full_features.loc[test_data.index]

# --------------------------------------------------
# ML inputs
# --------------------------------------------------
feature_cols = [
    c for c in test_df.columns
    if c not in ["target", "Open", "High", "Low", "Close", "Volume"]
]

X_test = test_df[feature_cols]
y_test = test_df["target"]
prices = test_df["Close"].values



Backtesting: NIFTY BANK
2025-12-30 15:45:56 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Nihar\Documents\GitHub\oop\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2025-12-30 15:45:58 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2025-12-30 15:45:58 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2025-12-30 15:45:58 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2025-12-30 15:45:58 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2025-12-30 15:45:58 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2025-12-30 15:45:58 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2025-12-30 15:45:58 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-01-09 09:15:00 to 2025-04-16 09:18:00
2025-12-30 15:

In [9]:
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------
# TRAIN FEATURES (DEFINE FEATURE SPACE HERE)
# --------------------------------------------------
train_with_signals = add_scalping_signals(train_data)
train_df = add_basic_features(train_with_signals)

feature_cols = [
    c for c in train_df.columns
    if c not in ["target", "Open", "High", "Low", "Close", "Volume"]
]

X_train = train_df[feature_cols]
y_train = train_df["target"]

# --------------------------------------------------
# SCALE (FIT ON TRAIN ONLY)
# --------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --------------------------------------------------
# MODEL
# --------------------------------------------------
model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=20,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    verbosity=0
)

model.fit(X_train_scaled, y_train)

# --------------------------------------------------
# ML PROBABILITIES
# --------------------------------------------------
ml_prob = model.predict_proba(X_test_scaled)[:, 1]
print("Test AUC:", roc_auc_score(y_test, ml_prob))


Test AUC: 0.6051143991247571


In [10]:
def position_size(prob, threshold):
    """
    Convert ML probability into position size [0, 1]
    """
    size = (prob - threshold) / (1 - threshold)
    return np.clip(size, 0, 1)

# ✅ COMPUTE ML_PROB FOR BACKTEST (using the newly trained model)
ml_prob = model.predict_proba(X_test_scaled)[:, 1]
print(f"\n✅ ML_PROB Generated from Trained Model:")
print(f"   Shape: {ml_prob.shape}")
print(f"   Min: {ml_prob.min():.4f}, Max: {ml_prob.max():.4f}, Mean: {ml_prob.mean():.4f}")



✅ ML_PROB Generated from Trained Model:
   Shape: (80907,)
   Min: 0.1003, Max: 0.5330, Mean: 0.2948


In [47]:

capital = INITIAL_CAPITAL
equity_curve = []
trades = []

# =========================================================
# COMPUTE ENTRY THRESHOLD
# =========================================================
print(f"ML Probability Distribution:")
print(f"  Min: {ml_prob.min():.4f}")
print(f"  25%: {np.percentile(ml_prob, 25):.4f}")
print(f"  Median: {np.median(ml_prob):.4f}")
print(f"  75%: {np.percentile(ml_prob, 75):.4f}")
print(f"  95%: {np.percentile(ml_prob, 95):.4f}")
print(f"  99%: {np.percentile(ml_prob, 99):.4f}")
print(f"  Max: {ml_prob.max():.4f}")

# Use top ENTRY_Q percentile
ENTRY_THRESHOLD = np.quantile(ml_prob, ENTRY_Q)
print(f"\n📊 Entry Threshold (Q={ENTRY_Q}): {ENTRY_THRESHOLD:.4f}")
print(f"📊 Potential signals above threshold: {(ml_prob >= ENTRY_THRESHOLD).sum()} / {len(ml_prob)}")

# =========================================================
# BACKTEST LOOP - NO TECHNICAL CONFIRMATION
# =========================================================
i = 0
n = len(prices)
filtered_out = 0

while i < n - HORIZON:

    prob = ml_prob[i]
    current_price = prices[i]
    
    # ==================== ENTRY FILTER 1: HIGH CONFIDENCE ONLY ====================
    if prob < ENTRY_THRESHOLD:
        equity_curve.append(capital)
        i += 1
        continue
    
    # ==================== ENTRY FILTER 2: REASONABLE CONFIDENCE ====================
    if prob > MAX_ML_CONFIDENCE:
        filtered_out += 1
        equity_curve.append(capital)
        i += 1
        continue
    
    # ==================== ENTRY FILTER 3: VOLATILITY GATE ====================
    vol = test_df["volatility_10"].iloc[i]
    vol = max(vol, 0.001)
    
    if vol > 0.05:
        equity_curve.append(capital)
        i += 1
        continue
    
    # ==================== POSITION SIZING ====================
    edge_strength = prob - ENTRY_THRESHOLD
    max_prob_range = 1.0 - ENTRY_THRESHOLD
    normalized_edge = edge_strength / max_prob_range
    
    size = np.clip(normalized_edge ** SIZE_EXPONENT, MIN_POSITION, MAX_POSITION)
    
    if size < MIN_POSITION:
        equity_curve.append(capital)
        i += 1
        continue
    
    position_value = capital * size
    entry_price = current_price
    
    # ==================== EXIT LOGIC ====================
    exit_price = prices[i + HORIZON]
    exit_idx = i + HORIZON
    exit_reason = "TIME"
    
    for j in range(1, HORIZON + 1):
        price = prices[i + j]
        
        if price <= entry_price * (1 - STOP_LOSS):
            exit_price = entry_price * (1 - STOP_LOSS)
            exit_idx = i + j
            exit_reason = "STOP"
            break
        
        if price >= entry_price * (1 + TAKE_PROFIT):
            exit_price = entry_price * (1 + TAKE_PROFIT)
            exit_idx = i + j
            exit_reason = "PROFIT"
            break
    
    # ==================== PnL CALCULATION ====================
    ret = (exit_price - entry_price) / entry_price
    net_ret = ret - (COST_PER_TRADE * 2)
    
    pnl = position_value * net_ret
    capital += pnl
    
    if capital < 0:
        capital = INITIAL_CAPITAL * 0.001
    
    trades.append({
        "entry_idx": i,
        "exit_idx": exit_idx,
        "prob": prob,
        "size": size,
        "return": net_ret,
        "pnl": pnl,
        "capital": capital,
        "exit_reason": exit_reason,
        "vol": vol
    })
    
    equity_curve.append(capital)
    
    i = exit_idx + COOLDOWN

print(f"\n✅ Backtest complete: {len(trades)} trades executed")
print(f"📊 Filtered out by confidence: {filtered_out} signals")


ML Probability Distribution:
  Min: 0.2286
  25%: 0.4191
  Median: 0.4791
  75%: 0.5362
  95%: 0.6102
  99%: 0.6463
  Max: 0.6830

📊 Entry Threshold (Q=0.65): 0.5120
📊 Potential signals above threshold: 28318 / 80907

✅ Backtest complete: 1300 trades executed
📊 Filtered out by confidence: 0 signals


In [14]:

# ======================================================
# RETRAIN MODEL WITH PATH 1 PARAMS FOR BACKTEST
# ======================================================
print("🔄 Retraining XGBoost with PATH 1 (Reduced Regularization)...")

from xgboost import XGBClassifier

# Prepare features
feature_cols = [
    col for col in train_df.columns
    if col not in ['target', 'Open', 'High', 'Low', 'Close', 'Volume', 'strategy_signal']
]

X_train_bt = train_df[feature_cols]
y_train_bt = train_df['target']
X_test_bt = test_df[feature_cols]

# Scale
from sklearn.preprocessing import StandardScaler
scaler_bt = StandardScaler()
X_train_scaled_bt = scaler_bt.fit_transform(X_train_bt)
X_test_scaled_bt = scaler_bt.transform(X_test_bt)

# Train with PATH 1 parameters
scale_pos_weight_bt = (len(X_train_scaled_bt) - y_train_bt.sum()) / max(y_train_bt.sum(), 1)

model_bt = XGBClassifier(
    n_estimators=200,
    max_depth=4,            # ✅ PATH 1: INCREASED from 3
    learning_rate=0.02,     # ✅ PATH 1: INCREASED from 0.01
    subsample=0.7,          # ✅ PATH 1: INCREASED from 0.5
    colsample_bytree=0.7,   # ✅ PATH 1: INCREASED from 0.5
    min_child_weight=10,    # ✅ PATH 1: REDUCED from 50
    gamma=0.1,              # ✅ PATH 1: REDUCED from 0.5
    reg_alpha=0.1,          # ✅ PATH 1: CRITICAL CHANGE (from 0.5)
    reg_lambda=0.5,         # ✅ PATH 1: CRITICAL CHANGE (from 3.0)
    scale_pos_weight=scale_pos_weight_bt,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    verbosity=0
)

model_bt.fit(X_train_scaled_bt, y_train_bt)

# Get ml_prob for backtest
ml_prob = model_bt.predict_proba(X_test_scaled_bt)[:, 1]

print(f"\n✅ PATH 1 MODEL READY FOR BACKTEST")
print(f"   Probability distribution:")
print(f"   Min: {ml_prob.min():.4f}, Max: {ml_prob.max():.4f}")
print(f"   Mean: {ml_prob.mean():.4f}, Median: {np.median(ml_prob):.4f}")
print(f"   THIS SHOULD BE MUCH BETTER THAN BEFORE (max was 0.533)")


🔄 Retraining XGBoost with PATH 1 (Reduced Regularization)...

✅ PATH 1 MODEL READY FOR BACKTEST
   Probability distribution:
   Min: 0.2286, Max: 0.6830
   Mean: 0.4762, Median: 0.4791
   THIS SHOULD BE MUCH BETTER THAN BEFORE (max was 0.533)


In [48]:
equity = pd.Series(equity_curve)
returns = equity.pct_change().dropna()

total_return = (equity.iloc[-1] / equity.iloc[0]) - 1
max_dd = ((equity / equity.cummax()) - 1).min()

# Proper intraday annualization
sharpe = (
    returns.mean() / returns.std()
    if returns.std() > 0 else 0
) * np.sqrt(252 * 6.5 * 60)

# Profit factor calculation
winning_trades = [t["pnl"] for t in trades if t["pnl"] > 0]
losing_trades = [t["pnl"] for t in trades if t["pnl"] < 0]

gross_profit = sum(winning_trades) if winning_trades else 0
gross_loss = abs(sum(losing_trades)) if losing_trades else 1e-8

profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf

win_rate = len(winning_trades) / len(trades) if trades else 0

# Expectancy calculation
avg_win = np.mean(winning_trades) if winning_trades else 0
avg_loss = np.mean(losing_trades) if losing_trades else 0
expectancy = (win_rate * avg_win) + ((1 - win_rate) * avg_loss)

print("\n" + "="*60)
print("🎯 BACKTEST RESULTS (OPTIMIZED FOR AUC 0.60 EDGE)")
print("="*60)
print(f"Initial Capital:   ₹{equity.iloc[0]:,.0f}")
print(f"Final Capital:     ₹{equity.iloc[-1]:,.0f}")
print(f"Total Return:      {total_return*100:.2f}%")
print(f"Net Profit:        ₹{equity.iloc[-1] - equity.iloc[0]:,.0f}")
print(f"Max Drawdown:      {max_dd*100:.2f}%")
print(f"Sharpe Ratio:      {sharpe:.2f}")
print(f"Profit Factor:     {profit_factor:.2f}")
print(f"Win Rate:          {win_rate*100:.2f}%")
print(f"Avg Win:           ₹{avg_win:,.0f}")
print(f"Avg Loss:          ₹{avg_loss:,.0f}")
print(f"Expectancy:        ₹{expectancy:,.0f} per trade")
print(f"Total Trades:      {len(trades)}")
print(f"Cost Per Trade:    {COST_PER_TRADE*100:.4f}% per side")
print("="*60)

# ==================== DIAGNOSTIC ANALYSIS ====================
print("\n" + "="*60)
print("🔍 DIAGNOSTIC REPORT")
print("="*60)

if len(trades) < 100:
    print(f"⚠️  WARNING: Only {len(trades)} trades - sample size too small")
    print("   Need 200+ trades for weak edge (AUC 0.60) to show in backtest")

# Check if cost is the problem
total_costs_paid = len(trades) * COST_PER_TRADE * 2 * INITIAL_CAPITAL / 1_000_000
print(f"📊 Total costs paid (estimate): ₹{total_costs_paid:,.0f}")
print(f"📊 Average profit needed per trade: ₹{total_return * INITIAL_CAPITAL / len(trades):,.0f}" if trades else "N/A")

if total_return <= 0 and len(trades) > 50:
    print("\n❌ STILL LOSING - Multiple issues:")
    
    if len(trades) < 150:
        print(f"   1. Volume too low ({len(trades)} trades)")
        print("      → Lower ENTRY_Q further (try 0.70)")
    
    if profit_factor < 1.0:
        print(f"   2. Win rate too low ({win_rate*100:.1f}%)")
        print("      → Increase STOP_LOSS (try 0.020)")
        print("      → Widen TAKE_PROFIT (try 0.030)")
    
    if avg_win <= abs(avg_loss):
        print(f"   3. Risk/reward imbalanced ({avg_win:.0f} win vs {avg_loss:.0f} loss)")
        print("      → Your model edge is weaker than expected")
        print("      → Consider retraining with different target")

elif total_return > 0:
    print(f"\n✅ PROFITABLE! +{total_return*100:.2f}% return on {len(trades)} trades")
    if profit_factor > 1.5:
        print("   Strategy is working - edge is clear")
    elif profit_factor > 1.0:
        print("   Strategy works but edge is small - scale up carefully")
else:
    print("\n⚠️  MARGINALLY PROFITABLE")
    print("   Continue testing with more data")



🎯 BACKTEST RESULTS (OPTIMIZED FOR AUC 0.60 EDGE)
Initial Capital:   ₹1,002,829
Final Capital:     ₹1,115,536
Total Return:      11.24%
Net Profit:        ₹112,707
Max Drawdown:      -9.33%
Sharpe Ratio:      1.52
Profit Factor:     1.06
Win Rate:          51.38%
Avg Win:           ₹3,038
Avg Loss:          ₹-3,028
Expectancy:        ₹89 per trade
Total Trades:      1300
Cost Per Trade:    0.0008% per side

🔍 DIAGNOSTIC REPORT
📊 Total costs paid (estimate): ₹0
📊 Average profit needed per trade: ₹86

✅ PROFITABLE! +11.24% return on 1300 trades
   Strategy works but edge is small - scale up carefully


In [98]:

# ======================================================
# CRITICAL: DIAGNOSE ML MODEL CALIBRATION
# ======================================================
print("\n" + "="*70)
print("🚨 ML MODEL CALIBRATION ANALYSIS")
print("="*70)

# Check actual trade distribution
trade_returns = [t["return"] for t in trades]

print(f"\n1️⃣  PROBABILITY DISTRIBUTION (Your model outputs):")
print(f"   Min: {ml_prob.min():.4f}")
print(f"   Max: {ml_prob.max():.4f}")
print(f"   Mean: {ml_prob.mean():.4f}")
print(f"   Median: {np.median(ml_prob):.4f}")
print(f"   Signals above 0.50 (neutral): {(ml_prob > 0.5).sum()} / {len(ml_prob)}")
print(f"   Signals above 0.60: {(ml_prob > 0.6).sum()} / {len(ml_prob)}")

if ml_prob.max() < 0.60:
    print("\n   ❌ PROBLEM: Model never outputs >60% confidence!")
    print("      This suggests the model is UNDERFITTED or REGULARIZED TOO HEAVILY")
    print("      → Check: max_depth, n_estimators, reg_alpha, reg_lambda")

if ml_prob.mean() < 0.40:
    print("\n   ⚠️  WARNING: Average confidence is {:.2%}".format(ml_prob.mean()))
    print("      Model probabilities are CALIBRATED to SELL signals (0 class)")
    print("      The target may be: does price go DOWN, not UP")

print(f"\n2️⃣  CLASS DISTRIBUTION (Training data):")
print(f"   Target 0 (no move): {(y_test == 0).sum()} samples")
print(f"   Target 1 (upside): {(y_test == 1).sum()} samples")
print(f"   Class ratio: {(y_test == 0).sum() / (y_test == 1).sum():.1f}:1")

if (y_test == 0).sum() / (y_test == 1).sum() > 10:
    print("   ❌ SEVERE CLASS IMBALANCE: >90% of samples are NEGATIVE class")
    print("      Model learns to predict 0 by default → all outputs cluster near 0")

print(f"\n3️⃣  TRADING RESULTS vs TARGET:")
print(f"   Actual trades executed: {len(trades)}")
print(f"   Win rate: {np.mean([1 for t in trades if t['return'] > 0]):.1%}")
print(f"   Expected win rate (for AUC 0.60): ~55%")

if np.mean([1 for t in trades if t['return'] > 0]) < 0.45:
    print("\n   ❌ WIN RATE BELOW 45%: Model predicts WORSE than random")
    print("      This is CONSISTENT with low probability outputs")
    print("      You're trading signals the model says are WEAK")

print(f"\n4️⃣  SOLUTION:")
print("   • Try using INVERSE predictions: entry when prob < 0.35")
print("   • Or retrain model with:")
print("      - Different target variable")
print("      - Less regularization (lower reg_alpha/reg_lambda)")
print("      - Different horizon (try 1, 5, 10, 20 bar horizons)")
print("="*70)

if not trades:
    print("\n❌ NO TRADES EXECUTED - Model is completely uncalibrated")



🚨 ML MODEL CALIBRATION ANALYSIS

1️⃣  PROBABILITY DISTRIBUTION (Your model outputs):
   Min: 0.2286
   Max: 0.6830
   Mean: 0.4762
   Median: 0.4791
   Signals above 0.50 (neutral): 32716 / 80907
   Signals above 0.60: 5504 / 80907

2️⃣  CLASS DISTRIBUTION (Training data):
   Target 0 (no move): 58125 samples
   Target 1 (upside): 22782 samples
   Class ratio: 2.6:1

3️⃣  TRADING RESULTS vs TARGET:
   Actual trades executed: 394
   Win rate: 100.0%
   Expected win rate (for AUC 0.60): ~55%

4️⃣  SOLUTION:
   • Try using INVERSE predictions: entry when prob < 0.35
   • Or retrain model with:
      - Different target variable
      - Less regularization (lower reg_alpha/reg_lambda)
      - Different horizon (try 1, 5, 10, 20 bar horizons)


In [99]:

# ======================================================
# DEBUG: SHOW ACTUAL TRADES TO FIND DISCREPANCY
# ======================================================
print("\nDEBUGGING TRADES:")
if trades:
    trades_df = pd.DataFrame(trades)
    print(f"\nFirst 5 trades:")
    print(trades_df[['prob', 'size', 'return', 'pnl', 'exit_reason']].head())
    print(f"\nLast 5 trades:")
    print(trades_df[['prob', 'size', 'return', 'pnl', 'exit_reason']].tail())
    
    print(f"\nReturn distribution:")
    print(f"  Positive returns: {(trades_df['return'] > 0).sum()}")
    print(f"  Zero/Negative returns: {(trades_df['return'] <= 0).sum()}")
    print(f"  Max return: {trades_df['return'].max():.6f}")
    print(f"  Min return: {trades_df['return'].min():.6f}")
    print(f"  Mean return: {trades_df['return'].mean():.6f}")
    
    print(f"\nPnL distribution (in rupees):")
    print(f"  Total PnL: ₹{trades_df['pnl'].sum():,.0f}")
    print(f"  Winning trades: {(trades_df['pnl'] > 0).sum()}")
    print(f"  Losing trades: {(trades_df['pnl'] < 0).sum()}")



DEBUGGING TRADES:

First 5 trades:
       prob  size    return         pnl exit_reason
0  0.314322  0.03  0.000774   23.219181        TIME
1  0.313706  0.03 -0.007862 -235.868902        TIME
2  0.314496  0.03 -0.000410  -12.283089        TIME
3  0.315829  0.03  0.001770   53.092579        TIME
4  0.329755  0.03  0.003667  110.000385        TIME

Last 5 trades:
         prob      size    return         pnl exit_reason
389  0.324637  0.030000  0.009602  285.516120        TIME
390  0.464797  0.150000 -0.006370 -947.347381        TIME
391  0.458843  0.150000  0.004079  606.102152        TIME
392  0.410912  0.142619  0.001555  219.815064        TIME
393  0.332194  0.030000 -0.001962  -58.364570        TIME

Return distribution:
  Positive returns: 176
  Zero/Negative returns: 218
  Max return: 0.024840
  Min return: -0.015160
  Mean return: -0.000111

PnL distribution (in rupees):
  Total PnL: ₹-8,693
  Winning trades: 176
  Losing trades: 218


In [14]:
import numpy as np
import pandas as pd

# ======================================================
# SCALPING FEATURES (STRATEGY + ML, LIVE-SAFE)
# ======================================================
def add_scalping_features(data, horizon=3, cost=0.0003, make_target=False):
    df = data.copy()

    # --------------------------------------------------
    # RETURNS
    # --------------------------------------------------
    df["returns"] = df["Close"].pct_change()
    df["log_returns"] = np.log(df["Close"] / df["Close"].shift(1))

    # --------------------------------------------------
    # TREND
    # --------------------------------------------------
    sma_10 = df["Close"].rolling(10).mean()
    sma_20 = df["Close"].rolling(20).mean()
    sma_50 = df["Close"].rolling(50).mean()

    df["trend_10"] = (df["Close"] - sma_10) / sma_10
    df["trend_20"] = (df["Close"] - sma_20) / sma_20
    df["trend_diff"] = (sma_10 - sma_20) / sma_20

    # --------------------------------------------------
    # PRICE ACTION
    # --------------------------------------------------
    df["range_pct"] = (df["High"] - df["Low"]) / df["Close"]
    df["body_pct"] = (df["Close"] - df["Open"]) / df["Close"]
    df["body_abs"] = df["body_pct"].abs()

    # --------------------------------------------------
    # VOLATILITY REGIME
    # --------------------------------------------------
    df["volatility_10"] = df["returns"].rolling(10).std()
    df["vol_ratio"] = df["volatility_10"] / df["volatility_10"].rolling(50).mean()
    df["high_vol"] = (df["vol_ratio"] > 1.0).astype(int)

    # --------------------------------------------------
    # RSI (0–1 scaled)
    # --------------------------------------------------
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    df["RSI"] = (100 - (100 / (1 + rs))) / 100.0

    # --------------------------------------------------
    # VOLUME
    # --------------------------------------------------
    if "Volume" in df.columns and df["Volume"].sum() > 0:
        vol_sma = df["Volume"].rolling(20).mean()
        df["Volume_norm"] = np.log1p(df["Volume"] / (vol_sma + 1e-8))
    else:
        df["Volume_norm"] = 0.0

    # --------------------------------------------------
    # RULE-BASED STRATEGY SIGNAL
    # --------------------------------------------------
    ema_12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema_26 = df["Close"].ewm(span=26, adjust=False).mean()
    macd = ema_12 - ema_26
    macd_signal = macd.ewm(span=9, adjust=False).mean()
    macd_hist = macd - macd_signal

    buy_uptrend = (df["Close"] > sma_20) & (sma_20 > sma_50)
    buy_rsi = df["RSI"] < 0.40
    buy_macd = (macd > 0) & (macd_hist > 0)

    close_20_high = df["Close"].rolling(20).max()
    buy_strength = df["Close"] > 0.95 * close_20_high

    sell_downtrend = (df["Close"] < sma_20) | (sma_20 < sma_50)
    sell_rsi = df["RSI"] > 0.60
    sell_macd = (macd < 0) & (macd_hist < 0)

    signal = pd.Series(0, index=df.index)
    signal[(buy_uptrend & buy_rsi) | (buy_uptrend & buy_macd) | (buy_uptrend & buy_strength)] = 1
    signal[(sell_downtrend & sell_rsi) | (sell_downtrend & sell_macd)] = -1

    df["strategy_signal"] = signal

    # --------------------------------------------------
    # TARGET (BACKTEST ONLY)
    # --------------------------------------------------
    if make_target:
        future_return = (df["Close"].shift(-horizon) - df["Close"]) / df["Close"]
        df["target"] = (future_return > cost).astype(int)

    return df

In [15]:
def fetch_intraday_data(period="1d"):
    df = yf.download(
        SYMBOL,
        period=period,
        interval="1m",
        progress=False
    )

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.columns = df.columns.astype(str).str.strip().str.capitalize()

    if "Adj close" in df.columns and "Close" not in df.columns:
        df.rename(columns={"Adj close": "Close"}, inplace=True)

    return df.dropna()


In [16]:
import yfinance as yf
SYMBOL = "^NSEBANK"


df_raw = fetch_intraday_data(period="1d")  # today (use "5d" if you want more)

df_feat = add_basic_features(
    df_raw,
    horizon=HORIZON,
    cost=COST_PER_TRADE
)

df_sig = add_scalping_signals(df_raw)
df_feat["strategy_signal"] = df_sig["strategy_signal"]

# Safety check
assert set(feature_cols).issubset(df_feat.columns)


In [17]:
paper_capital = INITIAL_CAPITAL
position = 0

entry_price = None
entry_time = None
qty = 0.0
invested_amount = 0.0

ENTRY_THRESHOLD = np.quantile(ml_prob, ENTRY_Q)  # FIXED: Use computed threshold
HORIZON = 12
TAKE_PROFIT = 0.0045
STOP_LOSS = 0.003

MIN_POSITION_FRACTION = 0.02
MAX_POSITION = 0.50
SIZE_EXPONENT = 2.0
COST_PER_TRADE = 0.0003

trades = []

for i in range(len(df_feat)):

    # Safe index check
    if i >= len(df_feat):
        break

    # Warm-up guard
    if i < 30:
        continue

    row = df_feat.iloc[i]
    ts = row.name
    current_price = row["Close"]

    # ML PREDICTION
    if feature_cols and all(c in row.index for c in feature_cols):
        X_live = scaler.transform(df_feat.iloc[[i]][feature_cols])
        prob = model.predict_proba(X_live)[0, 1]
    else:
        continue

    # ==============================
    # ENTRY (CORRECTED)
    # ==============================
    if position == 0 and prob >= ENTRY_THRESHOLD:

        position_fraction = min(
            (prob ** SIZE_EXPONENT) * MAX_POSITION,
            MAX_POSITION
        )

        if position_fraction < MIN_POSITION_FRACTION:
            continue

        invested_amount = paper_capital * position_fraction
        if invested_amount <= 0:
            continue

        qty = invested_amount / current_price

        # Lock capital with both costs
        paper_capital -= invested_amount * (1 + COST_PER_TRADE)

        entry_price = current_price
        entry_time = ts
        position = 1

        trades.append({
            "type": "BUY",
            "time": ts,
            "price": current_price,
            "prob": prob,
            "capital_used": invested_amount
        })

    # ==============================
    # EXIT (CORRECTED - add take profit)
    # ==============================
    elif position == 1:

        pnl_pct = (current_price - entry_price) / entry_price
        hold_minutes = int((ts - entry_time).total_seconds() // 60)

        should_exit = False
        exit_reason = ""

        if pnl_pct <= -STOP_LOSS:
            should_exit = True
            exit_reason = "STOP_LOSS"
        elif pnl_pct >= TAKE_PROFIT:
            should_exit = True
            exit_reason = "TAKE_PROFIT"
        elif hold_minutes >= HORIZON:
            should_exit = True
            exit_reason = "TIME_LIMIT"

        if should_exit:
            pnl_cash = (current_price - entry_price) * qty

            # Release capital and apply exit cost
            paper_capital += invested_amount + pnl_cash
            paper_capital -= invested_amount * COST_PER_TRADE

            trades.append({
                "type": "SELL",
                "time": ts,
                "price": current_price,
                "pnl_cash": pnl_cash,
                "pnl_pct": pnl_pct,
                "hold_min": hold_minutes,
                "exit_reason": exit_reason
            })

            position = 0
            entry_price = None
            entry_time = None
            qty = 0.0
            invested_amount = 0.0


In [18]:
trades_df = pd.DataFrame(trades)

if trades_df.empty:
    print("No trades executed.")
else:
    sell_trades = trades_df[trades_df["type"] == "SELL"].copy()
    
    print(f"Number of SELL trades: {len(sell_trades)}")
    
    if len(sell_trades) > 0:
        print(f"Avg PnL per trade: ₹{sell_trades['pnl_cash'].mean():.2f}")
        print(f"Total PnL: ₹{sell_trades['pnl_cash'].sum():.2f}")


Number of SELL trades: 1
Avg PnL per trade: ₹-25.72
Total PnL: ₹-25.72


In [19]:
if trades:
    total_trades = len([t for t in trades if t["type"] == "SELL"])
    
    sell_trades = [t for t in trades if t["type"] == "SELL"]
    
    if total_trades > 0:
        win_trades = [t for t in sell_trades if t["pnl_cash"] > 0]
        loss_trades = [t for t in sell_trades if t["pnl_cash"] <= 0]

        win_rate = len(win_trades) / total_trades if total_trades > 0 else 0.0

        avg_win = np.mean([t["pnl_cash"] for t in win_trades]) if win_trades else 0.0
        avg_loss = np.mean([t["pnl_cash"] for t in loss_trades]) if loss_trades else 0.0
        
        total_pnl = sum(t["pnl_cash"] for t in sell_trades)

        print("\n" + "="*50)
        print("PAPER TRADING SUMMARY (CORRECTED)")
        print("="*50)
        print(f"Total Trades:         {total_trades}")
        print(f"Win Rate:             {win_rate*100:.2f}%")
        print(f"Avg Win (₹):          {avg_win:.2f}")
        print(f"Avg Loss (₹):         {avg_loss:.2f}")
        print(f"Total PnL (₹):        {total_pnl:.2f}")
        print(f"Initial Capital:      {INITIAL_CAPITAL:,.2f}")
        print(f"Final Capital:        {paper_capital:,.2f}")
        print(f"Net Return:           {(paper_capital - INITIAL_CAPITAL)/INITIAL_CAPITAL * 100:.2f}%")
        print("="*50)
    else:
        print("No sell trades executed.")
else:
    print("No trades recorded.")



PAPER TRADING SUMMARY (CORRECTED)
Total Trades:         1
Win Rate:             0.00%
Avg Win (₹):          0.00
Avg Loss (₹):         -25.72
Total PnL (₹):        -25.72
Initial Capital:      1,000,000.00
Final Capital:        922,518.76
Net Return:           -7.75%


In [20]:
import yfinance as yf
import pandas as pd
import pytz
from datetime import datetime

IST = pytz.timezone("Asia/Kolkata")
SYMBOL = "^NSEBANK"

def fetch_today_data():
    df = yf.download(
        SYMBOL,
        period="1d",
        interval="1m",
        progress=False
    )

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.columns = df.columns.str.capitalize()

    # ---- timezone-safe handling
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")

    df.index = df.index.tz_convert(IST)

    # ---- FORCE only today's IST data
    today_ist = datetime.now(IST).date()
    df = df[df.index.date == today_ist]

    return df.dropna()
df_today = fetch_today_data()

num_days = df_today.index.normalize().nunique()
print("Number of unique days:", num_days)
df_today_feat = add_basic_features(
    df_today,
    horizon=HORIZON,
    cost=COST_PER_TRADE
)

# Optional if you use rule-based signals
df_today_feat = add_scalping_signals(df_today_feat)




Number of unique days: 1


In [21]:
# ===============================
# OFFLINE PAPER TRADING (REPLAY)
# ===============================

paper_capital = INITIAL_CAPITAL
position = 0

entry_price = None
entry_time = None
qty = 0.0
invested_amount = 0.0

ENTRY_THRESHOLD = 0.22
HORIZON = 12                     # minutes
TAKE_PROFIT = 0.0045             # 0.45%
STOP_LOSS = 0.003                # 0.30%

MIN_POSITION_FRACTION = 0.02
MAX_POSITION = 0.50              # max 50% capital
SIZE_EXPONENT = 2.0              # confidence scaling

trades = []

# ---- Replay ONLY on today's data
df_replay = df_today_feat.copy()

# ---- Sanity check
assert df_replay.index.normalize().nunique() == 1, "Replay is NOT 1-day data!"

for i in range(len(df_replay)):

    # Indicator warm-up (same as live)
    if i < 30:
        continue

    row = df_replay.iloc[i]
    ts = row.name
    current_price = row["Close"]

    # ==============================
    # ML PREDICTION
    # ==============================
    X_live = scaler.transform(df_replay.iloc[[i]][feature_cols])
    prob = model.predict_proba(X_live)[0, 1]

    # ==============================
    # ENTRY
    # ==============================
    if position == 0 and prob >= ENTRY_THRESHOLD:

        position_fraction = min(
            (prob ** SIZE_EXPONENT) * MAX_POSITION,
            MAX_POSITION
        )

        if position_fraction < MIN_POSITION_FRACTION:
            continue

        invested_amount = paper_capital * position_fraction
        qty = invested_amount / current_price

        # ---- lock capital
        paper_capital -= invested_amount
        paper_capital -= invested_amount * COST_PER_TRADE   # entry cost

        entry_price = current_price
        entry_time = ts
        position = 1

        trades.append({
            "type": "BUY",
            "time": ts,
            "price": current_price,
            "prob": prob,
            "capital_used": invested_amount
        })

    # ==============================
    # EXIT
    # ==============================
    elif position == 1:

        pnl_pct = (current_price - entry_price) / entry_price
        hold_minutes = int((ts - entry_time).total_seconds() // 60)

        if (
            pnl_pct <= -STOP_LOSS or
            pnl_pct >= TAKE_PROFIT or
            hold_minutes >= HORIZON
        ):

            pnl_cash = (current_price - entry_price) * qty

            # ---- release capital
            paper_capital += invested_amount
            paper_capital += pnl_cash
            paper_capital -= invested_amount * COST_PER_TRADE  # exit cost

            trades.append({
                "type": "SELL",
                "time": ts,
                "price": current_price,
                "pnl_cash": pnl_cash,
                "pnl_pct": pnl_pct,
                "hold_min": hold_minutes
            })

            # ---- reset state
            position = 0
            entry_price = None
            entry_time = None
            qty = 0.0
            invested_amount = 0.0


In [22]:
trades_df = pd.DataFrame(trades)
sell_trades = trades_df[trades_df["type"] == "SELL"]

print("Final Capital:", round(paper_capital, 2))
print("Total Trades:", len(sell_trades))
print("Total PnL:", sell_trades["pnl"].sum())

initial = INITIAL_CAPITAL
final = paper_capital

net_pnl = final - initial
net_return_pct = net_pnl / initial * 100

print(f"Initial Capital : {initial:,.2f}")
print(f"Final Capital   : {final:,.2f}")
print(f"Net PnL         : {net_pnl:,.2f}")
print(f"Net Return (%)  : {net_return_pct:.6f}%")


Final Capital: 953377.0
Total Trades: 14


KeyError: 'pnl'

In [ ]:
# ======================================================
# LIVE PAPER TRADING — 1 MIN INTRADAY (NIFTY BANK)
# CLEAN & CORRECTED
# ======================================================

import yfinance as yf
import time
from datetime import datetime, time as dtime
import pandas as pd
import pytz

# Note: This cell is for reference only. 
# Live trading requires:
# 1. Real-time data feed (yfinance is delayed)
# 2. Broker API integration
# 3. Proper error handling and monitoring
# 4. Paper trading account setup

print("""
╔════════════════════════════════════════════════════╗
║ LIVE TRADING TEMPLATE (DO NOT RUN)                 ║
║                                                    ║
║ Before deploying live, ensure:                     ║
║ ✓ Backtest shows consistent profitability          ║
║ ✓ Risk limits are set correctly                    ║
║ ✓ Real-time data source is configured              ║
║ ✓ Broker API authentication is set up              ║
║ ✓ Emergency exit procedures are in place           ║
╚════════════════════════════════════════════════════╝
""")

# This is a template - do not execute without proper infrastructure
# Instead, use paper trading above to validate the strategy

pass


ACTIVE SESSION: ^NSEBANK | LIVE PAPER TRADING
✅ SESSION ENDED


In [ ]:
# ======================================================
# PAPER TRADING PERFORMANCE ANALYSIS
# ======================================================

import pandas as pd
import numpy as np

paper_df = pd.DataFrame(paper_trades)

if paper_df.empty:
    print("No trades executed.")
else:
    # -------------------------------
    # Separate BUY / SELL trades
    # -------------------------------
    buys = paper_df[paper_df["side"] == "BUY"].reset_index(drop=True)
    sells = paper_df[paper_df["side"].isin(["SELL", "FORCED_SELL"])].reset_index(drop=True)

    trades = pd.concat([buys, sells], axis=1)
    trades = trades.loc[:, ~trades.columns.duplicated()]

    # -------------------------------
    # Returns
    # -------------------------------
    trades["return"] = trades["pnl"]
    trades.dropna(inplace=True)

    total_trades = len(trades)
    wins = trades[trades["return"] > 0]
    losses = trades[trades["return"] <= 0]

    win_rate = len(wins) / total_trades if total_trades > 0 else 0
    avg_win = wins["return"].mean() if not wins.empty else 0
    avg_loss = losses["return"].mean() if not losses.empty else 0

    profit_factor = (
        wins["return"].sum() / abs(losses["return"].sum())
        if not losses.empty else np.inf
    )

    # -------------------------------
    # Equity Curve & Drawdown
    # -------------------------------
    equity = paper_df["capital"].dropna()
    peak = equity.cummax()
    drawdown = (equity - peak) / peak

    max_dd = drawdown.min()

    # -------------------------------
    # Sharpe (intraday approx)
    # -------------------------------
    returns = trades["return"]
    sharpe = (
        np.sqrt(252 * 6.5 * 60) * returns.mean() / returns.std()
        if returns.std() != 0 else 0
    )

    # -------------------------------
    # Print Summary
    # -------------------------------
    print("=" * 60)
    print("INTRADAY PAPER TRADING SUMMARY")
    print("=" * 60)
    print(f"Final Capital      : ₹{paper_capital:,.0f}")
    print(f"Total Trades       : {total_trades}")
    print(f"Win Rate           : {win_rate*100:.2f}%")
    print(f"Avg Win            : {avg_win*100:.3f}%")
    print(f"Avg Loss           : {avg_loss*100:.3f}%")
    print(f"Profit Factor      : {profit_factor:.2f}")
    print(f"Max Drawdown       : {max_dd*100:.2f}%")
    print(f"Sharpe Ratio       : {sharpe:.2f}")
    print("=" * 60)

    trades


In [ ]:
paper_df = pd.DataFrame(paper_trades)
paper_df


In [ ]:
import os
from datetime import datetime

SAVE_DIR = "paper_trades"
os.makedirs(SAVE_DIR, exist_ok=True)

TODAY = datetime.now().strftime("%Y-%m-%d")
CSV_PATH = f"{SAVE_DIR}/niftybank_paper_{TODAY}.csv"

pd.DataFrame(paper_trades).to_csv(CSV_PATH, index=False)
pd.DataFrame(paper_trades).to_csv(CSV_PATH, index=False)

pd.DataFrame(paper_trades).to_csv(CSV_PATH, index=False)
print(f"Trades saved to {CSV_PATH}")
